## 1. Confirmar a GPU

In [1]:
!nvidia-smi

Thu Oct 29 17:54:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Conectar o Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Baixar o código e instalar a mesma versão da Ultralytics

In [3]:
!git clone https://github.com/candidoea/cv-inspecao-isoladores.git
%cd /content/cv-inspecao-isoladores
!git log --oneline -1
!pip install -q ultralytics==8.4.190

Cloning into 'cv-inspecao-isoladores'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 68 (delta 16), reused 62 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 319.87 KiB | 10.32 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/cv-inspecao-isoladores
b142a0e (HEAD -> main, origin/main, origin/HEAD) Fase 4: script de treino a partir de YAML com registro do experimento


## 4. Descompactar os dados

In [5]:
!mkdir -p data/processed
!unzip -q /content/drive/MyDrive/cv-inspecao-isoladores/processed.zip -d data/processed
!ls data/processed
!ls data/processed/train/images | wc -l

cplid.yaml  test  train  val
591


## 5. Reescrever o cplid.yaml com o caminho do Colab

In [6]:
from pathlib import Path

raiz = Path("data/processed").resolve()
conteudo = f"""path: {raiz.as_posix()}
train: train/images
val: val/images
test: test/images
names:
  0: isolador
  1: defeito
"""
(raiz / "cplid.yaml").write_text(conteudo, encoding="utf-8")
print(conteudo)

path: /content/cv-inspecao-isoladores/data/processed
train: train/images
val: val/images
test: test/images
names:
  0: isolador
  1: defeito



## 6. Teste rápido na GPU

In [7]:
!python src/train/treinar.py configs/exp01_baseline.yaml --epochs 1 --nome teste_colab

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
New https://pypi.org/project/ultralytics/8.4.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/cplid.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, 

## 7. Treino

In [8]:
!python src/train/treinar.py configs/exp01_baseline.yaml

New https://pypi.org/project/ultralytics/8.4.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/cplid.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.9

## 8. Salvar o resultado no Drive

In [9]:
!mkdir -p /content/drive/MyDrive/cv-inspecao-isoladores/runs
!cp -r runs/exp01_baseline /content/drive/MyDrive/cv-inspecao-isoladores/runs/
!ls /content/drive/MyDrive/cv-inspecao-isoladores/runs/exp01_baseline

args.yaml			 registro.json	      train_batch2.jpg
BoxF1_curve.png			 results.csv	      val_batch0_labels.jpg
BoxP_curve.png			 results.png	      val_batch0_pred.jpg
BoxPR_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxR_curve.png			 train_batch1480.jpg  val_batch1_pred.jpg
confusion_matrix_normalized.png  train_batch1481.jpg  val_batch2_labels.jpg
confusion_matrix.png		 train_batch1482.jpg  val_batch2_pred.jpg
labels.jpg			 train_batch1.jpg     weights
